In [4]:
import os
from typing import TypedDict, Literal
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

# Initialize the Groq model (Make sure GROQ_API_KEY is set in your environment)
# Using Llama 3.3 70B as it excels at complex reasoning and structural routing
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

# 1. The Shared Team Notebook (State)
class TeamState(TypedDict):
    task: str
    research_notes: str
    draft: str
    next_worker: str # The supervisor updates this to route the flow



In [5]:
# 2. Worker 1: The Researcher Agent
def researcher_node(state: TeamState):
    print("\n [Researcher] Searching the web and gathering facts...")
    
    system_prompt = (
        "You are an elite Research Analyst. Your goal is to find cold, hard facts, "
        "statistics, and technical realities about the topic provided. "
        "Do not write essays or introduce stylistic formatting. Provide raw, high-density bulleted facts."
    )
    
    user_prompt = f"Gather deep research, trends, and facts regarding this task: {state['task']}"
    
    # Fire call to Groq
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    
    return {"research_notes": response.content, "next_worker": "supervisor"}

# 3. Worker 2: The Writer Agent
def writer_node(state: TeamState):
    print("\n [Writer] Crafting a polished, high-engagement narrative...")
    
    system_prompt = (
        "You are a world-class Technical Content Marketer. Your job is to take raw research notes "
        "and transform them into highly engaging, beautifully structured LinkedIn posts or articles. "
        "Use short paragraphs, a captivating hook, bold text for key lines, and concise bullet points."
    )
    
    user_prompt = f"Write a compelling content piece about '{state['task']}' using ONLY these notes:\n\n{state['research_notes']}"
    
    # Fire call to Groq
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    
    return {"draft": response.content, "next_worker": "supervisor"}

In [6]:
# Define the strict output schema for the Supervisor
class RouteDecision(BaseModel):
    next_worker: Literal["researcher", "writer", "finish"] = Field(
        description="The next specialized agent to call, or 'finish' if the final draft is satisfactory."
    )
    reasoning: str = Field(description="A brief explanation of why this routing decision was made.")

# 4. The Supervisor Node
def supervisor_node(state: TeamState):
    print("\n [Supervisor] Reviewing project state and evaluating gaps...")
    
    system_prompt = (
        "You are an automated Project Manager supervising a data team. "
        "Your job is to look at what has been accomplished and decide who should work next.\n"
        "Guidelines:\n"
        "1. If 'research_notes' is completely missing or empty, assign 'researcher'.\n"
        "2. If 'research_notes' is present but 'draft' is missing or empty, assign 'writer'.\n"
        "3. If both research and an elegant draft are compiled, assign 'finish'."
    )
    
    # Presenting the current state variables to the supervisor
    current_state_brief = (
        f"Task Brief: {state.get('task')}\n"
        f"Has Research Notes? {'Yes' if state.get('research_notes') else 'No'}\n"
        f"Has Finished Draft? {'Yes' if state.get('draft') else 'No'}"
    )
    
    # Bind the structured output schema directly to Groq
    structured_llm = llm.with_structured_output(RouteDecision)
    
    decision: RouteDecision = structured_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=current_state_brief)
    ])
    
    print(f" Reason: {decision.reasoning}")
    print(f" Route Decision: Sending task to -> [{decision.next_worker.upper()}]")
    
    return {"next_worker": decision.next_worker}

In [7]:
# 5. Build and Stitch the Graph Workflows Together
builder = StateGraph(TeamState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "supervisor")

# Dynamic routing branch driven entirely by the Groq Supervisor's state output field
builder.add_conditional_edges(
    "supervisor",
    lambda state: state["next_worker"],
    {
        "researcher": "researcher",
        "writer": "writer",
        "finish": END
    }
)

# Workers loops back to the manager after execution
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")

# Compile the team app
team_app = builder.compile()
print(" Multi-Agent Brain Pipeline Successfully Compiled with Groq LLM integration!")

 Multi-Agent Brain Pipeline Successfully Compiled with Groq LLM integration!


In [8]:
# Provide the initial project brief
initial_brief = {
    "task": "Why developers are switching from linear agent frameworks to graph-based state machines in 2026",
    "research_notes": "",
    "draft": "",
    "next_worker": ""
}

print(" Waking up the agent team...")
final_output = team_app.invoke(initial_brief)

print("\n --- FINAL PRODUCT GENERATED BY GROQ SQUAD ---")
print(final_output["draft"])

 Waking up the agent team...

 [Supervisor] Reviewing project state and evaluating gaps...
 Reason: The task brief is missing research notes, so the researcher should work on it next.
 Route Decision: Sending task to -> [RESEARCHER]

 [Researcher] Searching the web and gathering facts...

 [Supervisor] Reviewing project state and evaluating gaps...
 Reason: Research notes are present, but a finished draft is missing.
 Route Decision: Sending task to -> [WRITER]

 [Writer] Crafting a polished, high-engagement narrative...

 [Supervisor] Reviewing project state and evaluating gaps...
 Reason: Both research notes and a finished draft are present, indicating that the task is complete.
 Route Decision: Sending task to -> [FINISH]

 --- FINAL PRODUCT GENERATED BY GROQ SQUAD ---
**The Future of Conversational AI: Why Developers Are Ditching Linear Agent Frameworks**

Are you tired of struggling with complex conversations in your chatbot development? You're not alone. In 2025, **75% of develop